In [ ]:
import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq

from deepeval.models import DeepEvalBaseLLM
from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase

load_dotenv()


class GroqLLM(DeepEvalBaseLLM):

    def __init__(self):
        self.model = ChatGroq(
            model="qwen/qwen3.6-27b",
            temperature=0,
            api_key=os.getenv("GROQ_API_KEY"),
        )

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        response = self.model.invoke(prompt)
        return response.content

    async def a_generate(self, prompt: str) -> str:
        response = await self.model.ainvoke(prompt)
        return response.content

    def get_model_name(self) -> str:
        return "qwen/qwen3.6-27b"


# Initialize DeepEval LLM
groq_model = GroqLLM()

# Faithfulness Metric
faithfulness = FaithfulnessMetric(
    threshold=0.7,
    model=groq_model,
)

# Example RAG output
result = {
    "answer": "Mahatma Gandhi was the leader of India's independence movement against British rule.",
    "retrieved_chunks": [
        {
            "text": (
                "Mahatma Gandhi, also known as Mohandas Karamchand Gandhi, "
                "led the Indian independence movement using nonviolent civil disobedience."
            )
        }
    ]
}

# Test Case
test_case = LLMTestCase(
    input="Who is Gandhi?",
    actual_output=result["answer"],
    retrieval_context=[
        chunk["text"]
        for chunk in result["retrieved_chunks"]
    ]
)

# Evaluate
faithfulness.measure(test_case)

print("=" * 50)
print("Model         :", groq_model.get_model_name())
print("Faithfulness  :", faithfulness.score)
print("Passed        :", faithfulness.success)
print("Reason        :", faithfulness.reason)

In [ ]:
# ---------------------------
# Generate JSON Report
# ---------------------------
report = {
    "metadata": {
        "timestamp": datetime.now().isoformat(),
        "model": groq_model.get_model_name(),
        "metric": "Faithfulness",
        "threshold": faithfulness.threshold,
    },
    "test_case": {
        "input": test_case.input,
        "actual_output": test_case.actual_output,
        "retrieval_context": retrieval_context,
    },
    "evaluation": {
        "score": faithfulness.score,
        "passed": faithfulness.success,
        "reason": faithfulness.reason,
    }
}

with open("faithfulness_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4, ensure_ascii=False)

print("✅ JSON report saved as 'faithfulness_report.json'")